# Website Contact Details Scraper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mamoo85/open-data-actor-notebooks/blob/main/notebooks/website-contact-extractor.ipynb)

**What:** Website contact extractor: get emails, phone numbers and social media profiles from any list of business websites. One clean row per site with role/named email tags, E.164 phones, 8 socials and address. $0.002 per website (5 pages per site by default); unreachable sites are free.

**Why:** run the [`website-contact-extractor`](https://apify.com/transparent_meteorite/website-contact-extractor) Apify actor from Python and analyse the result as a pandas DataFrame, with no scraping code to maintain.

> **Disclosure:** I am the author of this actor. It is pay-per-result; the free plan credit on a new Apify account covers a trial run like this one (this notebook uses a very small sample input).

**Privacy:** Actual email addresses, phone numbers and named contacts are NOT displayed. Only counts and company-level fields are shown, so no personal data is stored in this notebook.

## 1. Setup

In [1]:
!pip install -q apify-client pandas

import os, getpass
try:
    from google.colab import userdata
    APIFY_TOKEN = userdata.get('APIFY_TOKEN')          # Colab: add it under the key icon (Secrets)
except Exception:
    APIFY_TOKEN = os.environ.get('APIFY_TOKEN') or getpass.getpass('Apify API token: ')
assert APIFY_TOKEN, 'An Apify API token is required (a free account is enough).'
print('Token loaded.')

Token loaded.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Input
A deliberately small sample so the run is cheap and quick. See the [actor page](https://apify.com/transparent_meteorite/website-contact-extractor) for every option.

In [2]:
run_input = {   'startUrls': [{'url': 'https://www.shinola.com'}, {'url': 'https://buddyspizza.com'}],
    'maxPagesPerSite': 5,
    'includeNamedEmails': True,
    'sameDomainOnly': True}

## 3. Run the actor and read the dataset

In [3]:
from apify_client import ApifyClient

client = ApifyClient(APIFY_TOKEN)
run = client.actor('transparent_meteorite/website-contact-extractor').call(run_input=run_input)
# apify-client v2 returns a dict, v3 a Run object; handle both
get = (lambda k, a: run[k]) if isinstance(run, dict) else (lambda k, a: getattr(run, a))
items = list(client.dataset(get('defaultDatasetId', 'default_dataset_id')).iterate_items())
print('Run status:', get('status', 'status'), '| items:', len(items))

[apify.website-contact-extractor runId:cbsa45SO622kJqwVX] -> Status: RUNNING, Message: 


[apify.website-contact-extractor runId:cbsa45SO622kJqwVX] -> 2026-10-13T22:20:59.461Z ACTOR: Pulling container image of build 5IKGExIFAxYPOlpPK from registry.


[apify.website-contact-extractor runId:cbsa45SO622kJqwVX] -> 2026-10-13T22:20:59.465Z ACTOR: Creating container.


[apify.website-contact-extractor runId:cbsa45SO622kJqwVX] -> 2026-10-13T22:20:59.534Z ACTOR: Starting container.


[apify.website-contact-extractor runId:cbsa45SO622kJqwVX] -> 2026-10-13T22:20:59.535Z ACTOR: Running under "LIMITED_PERMISSIONS".


...

[apify.website-contact-extractor runId:cbsa45SO622kJqwVX] -> 2026-10-13T22:21:04.023Z Done: {"requested":2,"crawled":2,"charged":2,"withEmail":2,"withPhone":1,"skippedInvalid":0,"duplicates":0,"chargeLimitReached":false,"statusCounts":{"ok":2},"warnings":[]}


[apify.website-contact-extractor runId:cbsa45SO622kJqwVX] -> 2026-10-13T22:21:04.024Z INFO  [Status message]: Done: 2 items


[apify.website-contact-extractor runId:cbsa45SO622kJqwVX] -> 2026-10-13T22:21:04.069Z Thanks for using Website Contact Extractor. Found a bug or need a field? Open an issue on the Issues tab (https://console.apify.com/actors/zxuR66xK9DndMEUtr/issues) with the run URL and we will reply there.


[apify.website-contact-extractor runId:cbsa45SO622kJqwVX] -> Status: SUCCEEDED, Message: Done: 2 items


Run status: SUCCEEDED | items: 2


## 4. Tabulate

Actual email addresses, phone numbers and named contacts are NOT displayed. Only counts and company-level fields are shown, so no personal data is stored in this notebook.

In [4]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

df = pd.DataFrame(items)
df['hasContactPage'] = df['contactPageUrl'].notna()

# Privacy: remove contact / personal-data columns before anything is displayed
DROP = ["email", "phone", "author", "registeredagent", "contractor", "officer", "linkedin", "facebook", "instagram", "youtube", "tiktok", "socials", "twitter"]
KEEP = ["emailCount", "phoneCount", "emailProvider"]
df = df[[c for c in df.columns if c in KEEP or not any(p in c.lower() for p in DROP)]]
SHOW = [c for c in ["domain", "status", "companyName", "emailCount", "phoneCount", "hasContactPage", "pagesCrawled", "responded"] if c in df.columns]
df[SHOW]

,domain,status,companyName,emailCount,phoneCount,hasContactPage,pagesCrawled,responded
0,buddyspizza.com,ok,Buddy's Pizza,1,2,True,4,True
1,shinola.com,ok,Shinola,2,0,False,4,True


## 5. A quick analysis

In [5]:
df[['emailCount','phoneCount','pagesCrawled']]=df[['emailCount','phoneCount','pagesCrawled']].apply(pd.to_numeric,errors='coerce')
print('Share of sites exposing a contact page:', round(df['hasContactPage'].mean(),2))
df[['domain','emailCount','phoneCount','pagesCrawled']]

Share of sites exposing a contact page: 0.5


,domain,emailCount,phoneCount,pagesCrawled
0,buddyspizza.com,1,2,4
1,shinola.com,2,0,4


## Next steps

- Raise the limits in the input cell to run a larger sample.
- Schedule the actor or call it from your own pipeline: [website-contact-extractor on the Apify Store](https://apify.com/transparent_meteorite/website-contact-extractor).
- Check the Store page for the current pay-per-result price before scaling up.

Generated by `tools/generate.py`.